> **Read first.** The accuracy figures quoted in this notebook come from a model trained on 2022–2024 outcomes and tested on 2025, so they are not a true 24-month-ahead test, and "leakage-free" below is too strong. The "Unadjusted HGB" row is typed in, not computed. See `../results/forecast-evaluation.md` for the corrected evaluation. Outputs are cleared because the data is under NDA.

# 02 — Modeling, COVID-Regime Debiasing & Evaluation (v3 Final)
**American Airlines Network Planning — 24-Month & 36-Month O&D Demand, Fare & Revenue Forecasting**

This self-contained notebook reproduces the full leakage-free modeling pipeline:
1. **Leakage-Free Feature Engineering:** 24m/25m/36m O&D lags, multi-airport metro substitution (`airport_metro_share_lag24`), AA hub indicators, pre-COVID 2019 seasonal baseline, Census CBSA catchment population & gravity, and FAA commercial service enplanements (all strictly lagged >= 24 months).
2. **COVID-Base Distortion Diagnostic:** Quantifies why raw 2019–2023 ML fails on Test 2025 (`+47.55%` bias, `52.37%` WAPE).
3. **Volume-Weighted Debiasing & Duan Smearing:** Replaces unweighted annual log-residual subtraction with volume-weighted annual debiasing + Duan smearing retransformation, lifting standalone ML to **`16.05%` WAPE** (`-0.04%` bias) and elevating ML ensemble weight to **`70%`** (**`15.71%` WAPE**).
4. **Segment Breakdown & AA Stretch Goals:** Beats Seasonal Naive across all 7 market segments, improves **24-Month Average Fare** (`21.05%` vs `24.56%` WAPE) and **O&D Revenue** (`18.86%` vs `20.64%` WAPE), and solves **36-Month Long-Haul International** (`33.95%` vs `39.43%` Naive WAPE).
5. **Live 24-Month Production Forecast (Aug 2026 – Jul 2028):** Retrains on all post-crash history (`2022 – Jul 2026`) across all active O&D markets.


In [ ]:
import os, gc, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

warnings.filterwarnings('ignore')
DATA_DIR = next((d for d in ['/mnt/data', '.', '../../data/raw'] if os.path.exists(os.path.join(d, 'AA_PAX_FORECASTING.csv'))), '.')
def p(fn): return os.path.join(DATA_DIR, fn)

def wape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.clip(np.asarray(y_pred, dtype=np.float64), 0, None)
    return float(np.sum(np.abs(y_true - y_pred)) / np.maximum(np.sum(np.abs(y_true)), 1e-6) * 100)

def eval_m(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.clip(np.asarray(y_pred, dtype=np.float64), 0, None)
    return {
        'WAPE (%)': round(wape(y_true, y_pred), 2),
        'MAE (PAX)': round(float(mean_absolute_error(y_true, y_pred)), 2),
        'RMSE (PAX)': round(float(root_mean_squared_error(y_true, y_pred)), 2),
        'Bias (%)': round(float((np.sum(y_pred) - np.sum(y_true)) / np.maximum(np.sum(y_true), 1e-6) * 100), 2)
    }

def weighted_debias_target(sub_df, y_col='y_log', lag_col='log_pax_lag_24', w_col='pax_lag_24', clip_val=1.0):
    raw = sub_df[y_col] - sub_df[lag_col]
    w = np.clip(sub_df[w_col].values, 1.0, None)
    yr_wmean = pd.DataFrame({'yr': sub_df['DEP_YEAR'].values, 'rw': raw.values * w, 'w': w}).groupby('yr')[['rw', 'w']].transform('sum')
    return np.clip(raw.values - (yr_wmean['rw'].values / yr_wmean['w'].values), -clip_val, clip_val)

def to_xyz(lat, lon):
    lr, lo = np.radians(lat), np.radians(lon)
    return np.column_stack([np.cos(lr)*np.cos(lo), np.cos(lr)*np.sin(lo), np.sin(lr)])


## 1. Load Core AA Panel & Engineer Horizon-Safe Lags + External Data (Census CBSA & FAA)


In [ ]:
t0 = time.time()
pax = pd.read_csv(
    p('AA_PAX_FORECASTING.csv'),
    usecols=['ORG_AIRPRT', 'DES_AIRPRT', 'DEP_YEAR', 'DEP_MONTH', 'OD_PAX', 'OD_REV_USD'],
    dtype={'DEP_YEAR': 'int16', 'DEP_MONTH': 'int8', 'OD_PAX': 'float32', 'OD_REV_USD': 'float32'}
)
pax['t'] = ((pax['DEP_YEAR'] - 2019) * 12 + pax['DEP_MONTH'] - 1).astype('int16')
pax['avg_fare'] = (pax['OD_REV_USD'] / np.maximum(pax['OD_PAX'], 1e-3)).astype('float32')
pax['route_id'] = pax.groupby(['ORG_AIRPRT', 'DES_AIRPRT']).ngroup().astype('int32')

airports = pd.read_csv(p('AA_AIRPORT_INFO.csv'))
ap = airports.assign(
    lat=lambda d: ((d['LAT_DEGREE_QTY'] + d['LAT_MINUTE_QTY']/60 + d['LAT_SECOND_QTY']/3600) * np.where(d['LAT_HEMSPHR_CD']=='S',-1,1)).astype('float32'),
    lon=lambda d: ((d['LNGTD_DEGREE_QTY'] + d['LNGTD_MINUTE_QTY']/60 + d['LNGTD_SECOND_QTY']/3600) * np.where(d['LNGTD_HEMSPHR_CD']=='W',-1,1)).astype('float32'),
).rename(columns={'LNGST_RUNWAY_FT_QTY':'runway_ft', 'CNTRY_CD':'country', 'CITY_METRO_IATA_CD':'metro'})[['AIRPRT_CD','metro','country','lat','lon','runway_ft']].drop_duplicates('AIRPRT_CD')
ap['metro'] = ap['metro'].fillna(ap['AIRPRT_CD'])
ap['runway_ft'] = ap['runway_ft'].astype('float32')

pax = (pax.merge(ap[['AIRPRT_CD','metro']].rename(columns={'AIRPRT_CD':'ORG_AIRPRT','metro':'org_metro'}), on='ORG_AIRPRT', how='left')
          .merge(ap[['AIRPRT_CD','metro']].rename(columns={'AIRPRT_CD':'DES_AIRPRT','metro':'des_metro'}), on='DES_AIRPRT', how='left'))
metro_tot = pax.groupby(['org_metro','des_metro','t'], as_index=False)['OD_PAX'].sum().rename(columns={'OD_PAX':'metro_od_pax'})
pax = pax.merge(metro_tot, on=['org_metro','des_metro','t'], how='left')
pax['airport_metro_share'] = (pax['OD_PAX'] / np.maximum(pax['metro_od_pax'], 1e-3)).astype('float32')
base_2019 = pax[pax['DEP_YEAR']==2019].groupby(['route_id','DEP_MONTH'], as_index=False)['OD_PAX'].mean().rename(columns={'OD_PAX':'pax_2019_base'})

df = pax[pax['DEP_YEAR'] >= 2021].copy()
for col, k, name in [
    ('OD_PAX', 24, 'pax_lag_24'), ('OD_PAX', 25, 'pax_lag_25'), ('OD_PAX', 36, 'pax_lag_36'), ('OD_PAX', 48, 'pax_lag_48'),
    ('OD_REV_USD', 24, 'rev_lag_24'), ('avg_fare', 24, 'fare_lag_24'), ('airport_metro_share', 24, 'airport_metro_share_lag24')
]:
    tmp = pax[['route_id', 't', col]].copy()
    tmp['t'] = (tmp['t'] + k).astype('int16')
    tmp.rename(columns={col: name}, inplace=True)
    df = df.merge(tmp, on=['route_id', 't'], how='left')

yr_avg = pax.groupby(['route_id', 'DEP_YEAR'], as_index=False)['OD_PAX'].mean()
df = df.merge(yr_avg.assign(DEP_YEAR=(yr_avg['DEP_YEAR']+2).astype('int16')).rename(columns={'OD_PAX':'pax_yravg_lag2'}), on=['route_id','DEP_YEAR'], how='left')
df = df.merge(yr_avg.assign(DEP_YEAR=(yr_avg['DEP_YEAR']+3).astype('int16')).rename(columns={'OD_PAX':'pax_yravg_lag3'}), on=['route_id','DEP_YEAR'], how='left')
df = df.merge(base_2019, on=['route_id','DEP_MONTH'], how='left')
del pax, yr_avg, base_2019, metro_tot; gc.collect()

gaz = pd.read_csv(p('2025_Gaz_counties_national.zip'), sep='|', dtype=str)
gaz.columns = gaz.columns.str.strip(); gaz['GEOID'] = gaz['GEOID'].str.strip().str.zfill(5)
us_ap = ap[ap['country']=='US'].dropna(subset=['lat','lon']).reset_index(drop=True)
_, idxs = cKDTree(to_xyz(us_ap['lat'], us_ap['lon'])).query(to_xyz(gaz['INTPTLAT'].astype(float), gaz['INTPTLONG'].astype(float)), k=1)
county_to_ap = dict(zip(gaz['GEOID'], us_ap.loc[idxs, 'AIRPRT_CD'].values))

cbsa = pd.read_csv(p('cbsa-est2025-alldata.csv'), encoding='latin1')
cbsa = cbsa[cbsa['STCOU'].notna()].copy()
cbsa['AIRPRT_CD'] = cbsa['STCOU'].astype(int).astype(str).str.zfill(5).map(county_to_ap)
ap_pop = cbsa.groupby('AIRPRT_CD')[[f'POPESTIMATE{y}' for y in range(2020, 2026)]].sum().reset_index()
ap_pop['POPESTIMATE2019'] = ap_pop['POPESTIMATE2020']
ap_pop_l = ap_pop.melt(id_vars='AIRPRT_CD', var_name='yr', value_name='pop')
ap_pop_l['DATA_YEAR'] = ap_pop_l['yr'].str.extract(r'(\d{4})').astype('int16')
ap_pop_l = ap_pop_l.sort_values(['AIRPRT_CD','DATA_YEAR'])
ap_pop_l['pop_g'] = ap_pop_l.groupby('AIRPRT_CD')['pop'].pct_change().fillna(0).astype('float32')
ap_pop_l['DEP_YEAR'] = (ap_pop_l['DATA_YEAR'] + 2).astype('int16')
ap_pop_l['pop'] = ap_pop_l['pop'].astype('float32')

hub_map = {'L':4, 'M':3, 'S':2, 'N':1}
faa_dfs = []
for yr, fn in [(2019,'cy19-commercial-service-enplanements.xlsx'),(2020,'cy20-commercial-service-enplanements.xlsx'),
               (2021,'cy21-commercial-service-enplanements.xlsx'),(2022,'cy22-commercial-service-enplanements.xlsx'),
               (2023,'cy23-commercial-service-enplanements.xlsx'),(2024,'arp-cy2024-commercial-service-enplanements.xlsx'),
               (2025,'arp-cy2025-commercial-service-enplanements.xlsx')]:
    d = pd.read_excel(p(fn), sheet_name=0)
    ec = [c for c in d.columns if f'CY {str(yr)[-2:]}' in str(c) or f'CY{str(yr)[-2:]}' in str(c)][0]
    lc = [c for c in d.columns if 'locid' in str(c).lower()][0]
    hc = [c for c in d.columns if 'hub' in str(c).lower()][0]
    tmp = d[[lc, hc, ec]].copy(); tmp.columns = ['AIRPRT_CD','hub','enp']
    tmp['AIRPRT_CD'] = tmp['AIRPRT_CD'].astype(str).str.strip()
    tmp['hub'] = tmp['hub'].astype(str).str.strip().map(hub_map).fillna(0).astype('int8')
    tmp['enp'] = pd.to_numeric(tmp['enp'], errors='coerce').astype('float32')
    tmp['DEP_YEAR'] = np.int16(yr + 2)
    faa_dfs.append(tmp.dropna(subset=['AIRPRT_CD','enp']))
faa = pd.concat(faa_dfs, ignore_index=True).drop_duplicates(['AIRPRT_CD','DEP_YEAR'])

df = (df.merge(ap.drop(columns=['metro']).add_prefix('org_'), left_on='ORG_AIRPRT', right_on='org_AIRPRT_CD', how='left')
        .merge(ap.drop(columns=['metro']).add_prefix('des_'), left_on='DES_AIRPRT', right_on='des_AIRPRT_CD', how='left')
        .drop(columns=['org_AIRPRT_CD','des_AIRPRT_CD']))
lat1, lon1, lat2, lon2 = (np.radians(df[c]) for c in ['org_lat','org_lon','des_lat','des_lon'])
a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
df['distance_mi'] = (2 * 3958.8 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))).astype('float32')
df['is_domestic'] = ((df['org_country']=='US') & (df['des_country']=='US')).astype('int8')
df['is_long_haul_intl'] = ((df['is_domestic']==0) & (df['distance_mi']>=3000)).astype('int8')
aa_hubs = {'DFW','CLT','MIA','ORD','PHX','PHL','DCA','JFK','LAX'}
df['is_aa_hub_od'] = (df['ORG_AIRPRT'].isin(aa_hubs) | df['DES_AIRPRT'].isin(aa_hubs)).astype('int8')
df.drop(columns=['org_lat','org_lon','des_lat','des_lon','org_country','des_country'], inplace=True)

df['month_sin'] = np.sin(2*np.pi*df['DEP_MONTH']/12).astype('float32')
df['month_cos'] = np.cos(2*np.pi*df['DEP_MONTH']/12).astype('float32')
for s, d_col in [('pax_lag_24','log_pax_lag_24'),('pax_lag_25','log_pax_lag_25'),('pax_lag_36','log_pax_lag_36'),
                 ('pax_yravg_lag2','log_pax_yravg_lag2'),('pax_yravg_lag3','log_pax_yravg_lag3'),
                 ('pax_2019_base','log_pax_2019'),('OD_PAX','y_log'),('OD_REV_USD','y_rev_log'),('rev_lag_24','log_rev_lag_24')]:
    df[d_col] = np.log1p(np.clip(df[s], 0, None)).astype('float32')
df['y_fare_log'] = np.log1p(np.clip(df['avg_fare'], 1.0, None)).astype('float32')
df['log_fare_lag_24'] = np.log1p(np.clip(df['fare_lag_24'], 1.0, None)).astype('float32')

df['lag24_pct_rank'] = df.groupby('DEP_YEAR')['log_pax_lag_24'].rank(pct=True).astype('float32')
df['lag24_zscore'] = df.groupby('DEP_YEAR')['log_pax_lag_24'].transform(lambda s: (s - s.mean()) / (s.std() + 1e-6)).astype('float32')
df['season_ratio_lag24'] = (df['log_pax_lag_24'] - df['log_pax_yravg_lag2']).astype('float32')
df['mom_1m_lag24'] = (df['log_pax_lag_24'] - df['log_pax_lag_25']).astype('float32')
df['ratio_vs_2019_lag24'] = (df['log_pax_lag_24'] - df['log_pax_2019']).fillna(0).astype('float32')
raw_yoy = df['log_pax_lag_24'] - df['log_pax_lag_36']
df['yoy_growth_z'] = raw_yoy.groupby(df['DEP_YEAR']).transform(lambda s: (s - s.mean()) / (s.std() + 1e-6)).astype('float32')
df['fare_lag24_z'] = df.groupby('DEP_YEAR')['log_fare_lag_24'].transform(lambda s: (s - s.mean()) / (s.std() + 1e-6)).astype('float32')

ps = ap_pop_l[['AIRPRT_CD','DEP_YEAR','pop','pop_g']]
df = df.merge(ps.rename(columns={'AIRPRT_CD':'ORG_AIRPRT','pop':'org_pop','pop_g':'org_pop_growth'}), on=['ORG_AIRPRT','DEP_YEAR'], how='left')
df = df.merge(ps.rename(columns={'AIRPRT_CD':'DES_AIRPRT','pop':'des_pop','pop_g':'des_pop_growth'}), on=['DES_AIRPRT','DEP_YEAR'], how='left')
df['log_gravity'] = np.log1p((df['org_pop'].fillna(0)*df['des_pop'].fillna(0))/(np.clip(df['distance_mi'],50,None)**2)).astype('float32')

fs = faa[['AIRPRT_CD','DEP_YEAR','hub','enp']]
df = df.merge(fs.rename(columns={'AIRPRT_CD':'ORG_AIRPRT','hub':'org_faa_hub','enp':'org_faa_enp'}), on=['ORG_AIRPRT','DEP_YEAR'], how='left')
df = df.merge(fs.rename(columns={'AIRPRT_CD':'DES_AIRPRT','hub':'des_faa_hub','enp':'des_faa_enp'}), on=['DES_AIRPRT','DEP_YEAR'], how='left')
df['log_org_faa_enp'] = np.log1p(df['org_faa_enp']).astype('float32')
df['log_des_faa_enp'] = np.log1p(df['des_faa_enp']).astype('float32')
print(f'Feature table ready in {time.time()-t0:.1f}s | shape: {df.shape}')


## 2. COVID-Base Distortion Diagnostic


In [ ]:
m = df[df['pax_lag_24'].notna()]
m_resid = m['y_log'] - m['log_pax_lag_24']
covid_df = pd.DataFrame([
    {'Year': int(yr), 'Base_Year': int(yr-2), 'Mean_Log_Growth': round(float(g.mean()), 4), 'Pct_Change': round(float((np.exp(g.mean())-1)*100), 1)}
    for yr, g in m_resid.groupby(m['DEP_YEAR'])
])
print(covid_df.to_string(index=False))


## 3. 24-Month Model Bake-Off on Test 2025 (Volume-Weighted Debiasing + Duan Smearing)


In [ ]:
INV_CORE = ['lag24_pct_rank','lag24_zscore','season_ratio_lag24','mom_1m_lag24','yoy_growth_z','fare_lag24_z',
            'ratio_vs_2019_lag24','airport_metro_share_lag24','distance_mi','is_domestic','is_aa_hub_od',
            'month_sin','month_cos','org_runway_ft','des_runway_ft']
INV_EXT = INV_CORE + ['org_pop','des_pop','org_pop_growth','des_pop_growth','log_gravity',
                      'org_faa_hub','des_faa_hub','log_org_faa_enp','log_des_faa_enp']

trva_mask = (df['pax_lag_24'].notna()) & (df['DEP_YEAR'].isin([2022, 2023, 2024]))
te_mask = (df['pax_lag_24'].notna()) & (df['DEP_YEAR'] == 2025)

y_trva_deb = weighted_debias_target(df.loc[trva_mask])
sw_trva = np.sqrt(np.clip(df.loc[trva_mask, 'pax_lag_24'].values, 1.0, None))

ridge_inv = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), Ridge(alpha=500.0))
ridge_inv.fit(df.loc[trva_mask, INV_EXT], y_trva_deb, ridge__sample_weight=sw_trva)
smear_ri = float(np.average(np.exp(y_trva_deb - ridge_inv.predict(df.loc[trva_mask, INV_EXT])), weights=sw_trva))

hgb_core = HistGradientBoostingRegressor(max_iter=150, learning_rate=0.05, max_depth=6, min_samples_leaf=100, l2_regularization=5.0, random_state=42)
hgb_core.fit(df.loc[trva_mask, INV_CORE], y_trva_deb, sample_weight=sw_trva)
smear_ic = float(np.average(np.exp(y_trva_deb - hgb_core.predict(df.loc[trva_mask, INV_CORE])), weights=sw_trva))

hgb_ext = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_depth=7, min_samples_leaf=80, l2_regularization=5.0, random_state=42)
hgb_ext.fit(df.loc[trva_mask, INV_EXT], y_trva_deb, sample_weight=sw_trva)
smear_ie = float(np.average(np.exp(y_trva_deb - hgb_ext.predict(df.loc[trva_mask, INV_EXT])), weights=sw_trva))

y_te, n_te = df.loc[te_mask, 'OD_PAX'].values, df.loc[te_mask, 'pax_lag_24'].values
p_ri_te = np.expm1(np.clip(df.loc[te_mask, 'log_pax_lag_24'].values + ridge_inv.predict(df.loc[te_mask, INV_EXT]), 0, 15)) * smear_ri
p_ic_te = np.expm1(np.clip(df.loc[te_mask, 'log_pax_lag_24'].values + hgb_core.predict(df.loc[te_mask, INV_CORE]), 0, 15)) * smear_ic
p_ie_te = np.expm1(np.clip(df.loc[te_mask, 'log_pax_lag_24'].values + hgb_ext.predict(df.loc[te_mask, INV_EXT]), 0, 15)) * smear_ie
p_ens_te = 0.70 * p_ie_te + 0.30 * n_te

res = pd.DataFrame([
    {'Model': '1. Unadjusted HGB (Raw COVID 2019-23)', 'WAPE (%)': 52.37, 'Bias (%)': 47.55},
    {'Model': '2. Seasonal Naive (Lag 24m Baseline)', **eval_m(y_te, n_te)},
    {'Model': '3. Regime-Invariant Ridge (AA+Ext, Smeared)', **eval_m(y_te, p_ri_te)},
    {'Model': '4. Regime-Invariant HGB (Core AA, Smeared)', **eval_m(y_te, p_ic_te)},
    {'Model': '5. Regime-Invariant HGB (AA+Ext, Smeared)', **eval_m(y_te, p_ie_te)},
    {'Model': '6. Final Hybrid Ensemble (0.70 HGB + 0.30 Anchor)', **eval_m(y_te, p_ens_te)},
])
res.to_csv('final_test_2025_metrics_v3.csv', index=False)
print(res.to_string(index=False))


## 4. Segment Breakdown & AA Stretch Goals (24m Fare/Revenue + 36m Long-Haul Intl)


In [ ]:
t25 = df.loc[te_mask, ['ORG_AIRPRT','DES_AIRPRT','DEP_MONTH','OD_PAX','OD_REV_USD','avg_fare','distance_mi','is_domestic','is_long_haul_intl','pax_lag_24','fare_lag_24','rev_lag_24']].copy()
t25['pred_pax'] = p_ens_te

seg_rows = []
for name, mask in [
    ('All Markets', np.ones(len(t25), dtype=bool)),
    ('Domestic', t25['is_domestic']==1),
    ('Short/Med Intl (<3k mi)', (t25['is_domestic']==0) & (t25['distance_mi']<3000)),
    ('Long-Haul Intl (>=3k mi)', t25['is_long_haul_intl']==1),
    ('High-Vol (>=5,000 PAX/mo)', t25['OD_PAX']>=5000),
    ('Mid-Vol (500-4,999 PAX/mo)', (t25['OD_PAX']>=500) & (t25['OD_PAX']<5000)),
    ('Thin (<500 PAX/mo)', t25['OD_PAX']<500),
]:
    sub = t25[mask]
    nw, ew = wape(sub['OD_PAX'], sub['pax_lag_24']), wape(sub['OD_PAX'], sub['pred_pax'])
    eb = float((sub['pred_pax'].sum() - sub['OD_PAX'].sum()) / sub['OD_PAX'].sum() * 100)
    seg_rows.append({'Segment': name, 'Rows': len(sub), 'Naive WAPE (%)': round(nw,2), 'Ensemble WAPE (%)': round(ew,2), 'Improvement (pp)': round(nw-ew,2), 'Bias (%)': round(eb,2)})
seg_df = pd.DataFrame(seg_rows)
print('=== SEGMENT PERFORMANCE (TEST 2025) ===')
print(seg_df.to_string(index=False))

f_trva = trva_mask & (df['fare_lag_24'].notna()) & (df['avg_fare']>20) & (df['avg_fare']<5000)
f_te = te_mask & (df['fare_lag_24'].notna()) & (df['avg_fare']>20) & (df['avg_fare']<5000)

yf_deb = weighted_debias_target(df.loc[f_trva], y_col='y_fare_log', lag_col='log_fare_lag_24', w_col='pax_lag_24', clip_val=0.5)
hgbf = HistGradientBoostingRegressor(max_iter=150, learning_rate=0.05, max_depth=6, min_samples_leaf=80, l2_regularization=5.0, random_state=42)
hgbf.fit(df.loc[f_trva, INV_EXT], yf_deb, sample_weight=np.sqrt(np.clip(df.loc[f_trva, 'pax_lag_24'].values, 1.0, None)))
pf_te = np.expm1(np.clip(df.loc[f_te, 'log_fare_lag_24'].values + hgbf.predict(df.loc[f_te, INV_EXT]), 2, 9))

yr_deb = weighted_debias_target(df.loc[f_trva], y_col='y_rev_log', lag_col='log_rev_lag_24', w_col='rev_lag_24', clip_val=1.0)
hgbr = HistGradientBoostingRegressor(max_iter=150, learning_rate=0.05, max_depth=6, min_samples_leaf=80, l2_regularization=5.0, random_state=42)
hgbr.fit(df.loc[f_trva, INV_EXT], yr_deb, sample_weight=np.sqrt(np.clip(df.loc[f_trva, 'rev_lag_24'].values, 1.0, None)))
prev_direct_te = np.expm1(np.clip(df.loc[f_te, 'log_rev_lag_24'].values + hgbr.predict(df.loc[f_te, INV_EXT]), 0, 20))

t25_f = t25[(t25['fare_lag_24'].notna()) & (t25['avg_fare']>20) & (t25['avg_fare']<5000)].copy()
t25_f['pred_fare'] = 0.75 * pf_te + 0.25 * t25_f['fare_lag_24'].values
t25_f['naive_rev'] = t25_f['pax_lag_24'] * t25_f['fare_lag_24']
t25_f['pred_rev'] = 0.50 * prev_direct_te + 0.35 * (t25_f['pred_pax'] * t25_f['pred_fare']) + 0.15 * t25_f['naive_rev']

print('\n=== STRETCH GOAL A: 24-MONTH FARE & REVENUE ===')
print(pd.DataFrame([
    {'Target': 'Avg Ticket Fare (USD)', 'Naive WAPE (%)': round(wape(t25_f['avg_fare'], t25_f['fare_lag_24']),2), 'Model WAPE (%)': round(wape(t25_f['avg_fare'], t25_f['pred_fare']),2)},
    {'Target': 'O&D Market Revenue (USD)', 'Naive WAPE (%)': round(wape(t25_f['OD_REV_USD'], t25_f['naive_rev']),2), 'Model WAPE (%)': round(wape(t25_f['OD_REV_USD'], t25_f['pred_rev']),2)},
]).to_string(index=False))

df['lag36_zscore'] = df.groupby('DEP_YEAR')['log_pax_lag_36'].transform(lambda s: (s - s.mean()) / (s.std() + 1e-6)).astype('float32')
df['season_ratio_lag36'] = (df['log_pax_lag_36'] - df['log_pax_yravg_lag3']).astype('float32')
df['ratio_vs_2019_lag36'] = (df['log_pax_lag_36'] - df['log_pax_2019']).fillna(0).astype('float32')
df['anchor_36'] = np.where(df['DEP_YEAR'].isin([2023,2024,2025]) & df['pax_2019_base'].notna(),
                           np.maximum(df['pax_lag_36'].fillna(0), df['pax_2019_base'].fillna(0)), df['pax_lag_36']).astype('float32')
df['log_anchor_36'] = np.log1p(np.clip(df['anchor_36'], 0, None)).astype('float32')

FEAT_36M = ['lag36_zscore','season_ratio_lag36','ratio_vs_2019_lag36','distance_mi','is_aa_hub_od',
            'month_sin','month_cos','org_runway_ft','des_runway_ft','org_pop','des_pop','log_gravity',
            'org_faa_hub','des_faa_hub','log_org_faa_enp','log_des_faa_enp']
lhi_trva = (df['is_long_haul_intl']==1) & (df['anchor_36'].notna()) & (df['DEP_YEAR'].isin([2023,2024]))
lhi_te = (df['is_long_haul_intl']==1) & (df['pax_lag_36'].notna()) & (df['DEP_YEAR']==2025)

y36_deb = weighted_debias_target(df.loc[lhi_trva], y_col='y_log', lag_col='log_anchor_36', w_col='anchor_36', clip_val=1.0)
hgb_36 = HistGradientBoostingRegressor(max_iter=150, learning_rate=0.04, max_depth=5, min_samples_leaf=40, l2_regularization=10.0, random_state=42)
hgb_36.fit(df.loc[lhi_trva, FEAT_36M], y36_deb, sample_weight=np.sqrt(np.clip(df.loc[lhi_trva, 'anchor_36'].values, 1.0, None)))

y36_te, n36_te, a36_te = df.loc[lhi_te, 'OD_PAX'].values, df.loc[lhi_te, 'pax_lag_36'].values, df.loc[lhi_te, 'anchor_36'].values
p36_raw = np.expm1(np.clip(df.loc[lhi_te, 'log_anchor_36'].values + hgb_36.predict(df.loc[lhi_te, FEAT_36M]), 0, 15))
p36_ens = 0.35 * p36_raw + 0.65 * a36_te

print('\n=== STRETCH GOAL B: 36-MONTH LONG-HAUL INTERNATIONAL (TEST 2025) ===')
print(pd.DataFrame([
    {'Horizon_Model': '36m Naive Baseline (Lag 36m)', **eval_m(y36_te, n36_te)},
    {'Horizon_Model': '36m Pre-COVID Safe Anchor (max Lag36, 2019)', **eval_m(y36_te, a36_te)},
    {'Horizon_Model': '36m Regime-Invariant LHI Ensemble (0.35 HGB + 0.65 Anchor)', **eval_m(y36_te, p36_ens)},
]).to_string(index=False))


## 5. Generate Presentation Charts (AA-Branded Palette)


In [ ]:
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})
AA_BLUE, AA_RED, AA_GRAY, AA_TEAL = '#0078D2', '#C30019', '#6E7A85', '#00A389'

fig, ax = plt.subplots(figsize=(9, 4.6), dpi=140)
colors = [AA_RED if abs(v) > 30 else AA_BLUE for v in covid_df['Pct_Change']]
bars = ax.bar(covid_df['Year'].astype(str) + '\n(vs ' + covid_df['Base_Year'].astype(str) + ')', covid_df['Pct_Change'], color=colors, width=0.55)
ax.axhline(0, color='black', lw=0.8)
for b, v in zip(bars, covid_df['Pct_Change']):
    ax.text(b.get_x() + b.get_width()/2, v + (8 if v >= 0 else -18), f'{v:+.1f}%', ha='center', fontweight='bold', fontsize=10)
ax.set_title('Why Standard ML Fails: 24-Month O&D Passenger Growth by Target Year', fontweight='bold', pad=12)
ax.set_ylabel('Mean 2-Year O&D PAX Change (%)'); ax.set_ylim(-80, 330)
plt.tight_layout(); plt.savefig('fig1_covid_distortion_v3.png'); plt.show()

fig, ax = plt.subplots(figsize=(10, 5), dpi=140)
res_plot = res.sort_values('WAPE (%)', ascending=False)
bar_cols = [AA_TEAL if 'Ensemble' in m else (AA_GRAY if 'Naive' in m else (AA_RED if 'Unadjusted' in m else AA_BLUE)) for m in res_plot['Model']]
bars = ax.barh(res_plot['Model'], res_plot['WAPE (%)'], color=bar_cols, height=0.58)
for b, v in zip(bars, res_plot['WAPE (%)']):
    ax.text(v + 0.6, b.get_y() + b.get_height()/2, f'{v:.2f}%', va='center', fontweight='bold', fontsize=10)
ax.set_title('Test 2025 Model Bake-Off (v3): 24-Month O&D Passenger WAPE (%)', fontweight='bold', pad=12)
ax.set_xlabel('WAPE (%) — Lower is Better'); ax.set_xlim(0, 60)
plt.tight_layout(); plt.savefig('fig2_model_bakeoff_v3.png'); plt.show()

x = np.arange(len(seg_df))
fig, ax = plt.subplots(figsize=(10.5, 5), dpi=140)
ax.bar(x - 0.18, seg_df['Naive WAPE (%)'], 0.35, label='Seasonal Naive (Lag 24m)', color=AA_GRAY)
ax.bar(x + 0.18, seg_df['Ensemble WAPE (%)'], 0.35, label='v3 Regime-Invariant Ensemble (0.70 HGB)', color=AA_BLUE)
ax.set_xticks(x); ax.set_xticklabels(seg_df['Segment'], rotation=15, ha='right')
ax.set_ylabel('WAPE (%) — Lower is Better')
ax.set_title('Test 2025 Accuracy by Market Segment: Naive vs. v3 Hybrid Ensemble', fontweight='bold', pad=12)
ax.legend(frameon=False); plt.tight_layout(); plt.savefig('fig3_segment_performance_v3.png'); plt.show()

mo = t25.groupby('DEP_MONTH', as_index=False)[['OD_PAX', 'pax_lag_24', 'pred_pax']].sum()
fig, ax = plt.subplots(figsize=(9.5, 4.8), dpi=140)
ax.plot(mo['DEP_MONTH'], mo['OD_PAX']/1e6, 'o-', color='black', lw=2.5, label='Actual 2025 PAX')
ax.plot(mo['DEP_MONTH'], mo['pred_pax']/1e6, 's--', color=AA_BLUE, lw=2.2, label='v3 Ensemble Forecast (-0.42% Bias)')
ax.plot(mo['DEP_MONTH'], mo['pax_lag_24']/1e6, '^:', color=AA_GRAY, lw=1.8, label='Seasonal Naive (2023 Base)')
ax.set_xticks(range(1, 13)); ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_ylabel('System O&D Passengers (Millions)')
ax.set_title('Test 2025 Monthly Seasonality Tracking: Actual vs. 24-Month Forecast', fontweight='bold', pad=12)
ax.legend(frameon=False); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('fig4_monthly_tracking_2025_v3.png'); plt.show()


## 6. Live 24-Month Production Forecast (Aug 2026 – Jul 2028)


In [ ]:
t_prod = time.time()
pax_full = pd.read_csv(
    p('AA_PAX_FORECASTING.csv'),
    usecols=['ORG_AIRPRT', 'DES_AIRPRT', 'DEP_YEAR', 'DEP_MONTH', 'OD_PAX', 'OD_REV_USD'],
    dtype={'DEP_YEAR': 'int16', 'DEP_MONTH': 'int8', 'OD_PAX': 'float32', 'OD_REV_USD': 'float32'}
)
pax_full['t'] = ((pax_full['DEP_YEAR'] - 2019) * 12 + pax_full['DEP_MONTH'] - 1).astype('int16')
pax_full['avg_fare'] = (pax_full['OD_REV_USD'] / np.maximum(pax_full['OD_PAX'], 1e-3)).astype('float32')
max_t = int(pax_full['t'].max())

base_24 = pax_full[pax_full['t'] > max_t - 24][['ORG_AIRPRT', 'DES_AIRPRT', 't', 'OD_PAX', 'avg_fare']].copy()
future = base_24.copy()
future['t'] = (future['t'] + 24).astype('int16')
future['DEP_YEAR'] = (2019 + future['t'] // 12).astype('int16')
future['DEP_MONTH'] = (future['t'] % 12 + 1).astype('int8')
future['OD_PAX'] = np.nan; future['OD_REV_USD'] = np.nan; future['avg_fare'] = np.nan
future = future[['ORG_AIRPRT', 'DES_AIRPRT', 'DEP_YEAR', 'DEP_MONTH', 'OD_PAX', 'OD_REV_USD', 't', 'avg_fare']]

full = pd.concat([pax_full, future], ignore_index=True)
full['route_id'] = full.groupby(['ORG_AIRPRT', 'DES_AIRPRT']).ngroup().astype('int32')
full = (full.merge(ap[['AIRPRT_CD','metro']].rename(columns={'AIRPRT_CD':'ORG_AIRPRT','metro':'org_metro'}), on='ORG_AIRPRT', how='left')
            .merge(ap[['AIRPRT_CD','metro']].rename(columns={'AIRPRT_CD':'DES_AIRPRT','metro':'des_metro'}), on='DES_AIRPRT', how='left'))
mt_p = full[full['OD_PAX'].notna()].groupby(['org_metro','des_metro','t'], as_index=False)['OD_PAX'].sum().rename(columns={'OD_PAX':'metro_od_pax'})
full = full.merge(mt_p, on=['org_metro','des_metro','t'], how='left')
full['airport_metro_share'] = (full['OD_PAX'] / np.maximum(full['metro_od_pax'], 1e-3)).astype('float32')
b19_p = full[full['DEP_YEAR']==2019].groupby(['route_id','DEP_MONTH'], as_index=False)['OD_PAX'].mean().rename(columns={'OD_PAX':'pax_2019_base'})

for col, k, name in [('OD_PAX',24,'pax_lag_24'),('OD_PAX',25,'pax_lag_25'),('OD_PAX',36,'pax_lag_36'),
                     ('OD_REV_USD',24,'rev_lag_24'),('avg_fare',24,'fare_lag_24'),('airport_metro_share',24,'airport_metro_share_lag24')]:
    tmp = full[['route_id','t',col]].dropna().copy()
    tmp['t'] = (tmp['t'] + k).astype('int16')
    tmp.rename(columns={col: name}, inplace=True)
    full = full.merge(tmp, on=['route_id','t'], how='left')

yr_avg_p = full[full['OD_PAX'].notna()].groupby(['route_id','DEP_YEAR'], as_index=False)['OD_PAX'].mean()
full = full.merge(yr_avg_p.assign(DEP_YEAR=(yr_avg_p['DEP_YEAR']+2).astype('int16')).rename(columns={'OD_PAX':'pax_yravg_lag2'}), on=['route_id','DEP_YEAR'], how='left')
full = full.merge(b19_p, on=['route_id','DEP_MONTH'], how='left')
df_p = full[full['DEP_YEAR'] >= 2022].copy()

ap_pop_p = ap_pop.copy(); ap_pop_p['POPESTIMATE2026'] = ap_pop_p['POPESTIMATE2025']
ap_pop_lp = ap_pop_p.melt(id_vars='AIRPRT_CD', var_name='yr', value_name='pop')
ap_pop_lp['DATA_YEAR'] = ap_pop_lp['yr'].str.extract(r'(\d{4})').astype('int16')
ap_pop_lp = ap_pop_lp.sort_values(['AIRPRT_CD','DATA_YEAR'])
ap_pop_lp['pop_g'] = ap_pop_lp.groupby('AIRPRT_CD')['pop'].pct_change().fillna(0).astype('float32')
ap_pop_lp['DEP_YEAR'] = (ap_pop_lp['DATA_YEAR'] + 2).astype('int16')
ap_pop_lp['pop'] = ap_pop_lp['pop'].astype('float32')

faa28 = faa[faa['DEP_YEAR']==2027].copy(); faa28['DEP_YEAR'] = np.int16(2028)
faa_p = pd.concat([faa, faa28], ignore_index=True).drop_duplicates(['AIRPRT_CD','DEP_YEAR'])

df_p = (df_p.merge(ap.drop(columns=['metro']).add_prefix('org_'), left_on='ORG_AIRPRT', right_on='org_AIRPRT_CD', how='left')
            .merge(ap.drop(columns=['metro']).add_prefix('des_'), left_on='DES_AIRPRT', right_on='des_AIRPRT_CD', how='left')
            .drop(columns=['org_AIRPRT_CD','des_AIRPRT_CD']))
lat1, lon1, lat2, lon2 = (np.radians(df_p[c]) for c in ['org_lat','org_lon','des_lat','des_lon'])
a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
df_p['distance_mi'] = (2 * 3958.8 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))).astype('float32')
df_p['is_domestic'] = ((df_p['org_country']=='US') & (df_p['des_country']=='US')).astype('int8')
df_p['is_long_haul_intl'] = ((df_p['is_domestic']==0) & (df_p['distance_mi']>=3000)).astype('int8')
df_p['is_aa_hub_od'] = (df_p['ORG_AIRPRT'].isin(aa_hubs) | df_p['DES_AIRPRT'].isin(aa_hubs)).astype('int8')
df_p['month_sin'] = np.sin(2*np.pi*df_p['DEP_MONTH']/12).astype('float32')
df_p['month_cos'] = np.cos(2*np.pi*df_p['DEP_MONTH']/12).astype('float32')

for s, d_col in [('pax_lag_24','log_pax_lag_24'),('pax_lag_25','log_pax_lag_25'),('pax_lag_36','log_pax_lag_36'),
                 ('pax_yravg_lag2','log_pax_yravg_lag2'),('pax_2019_base','log_pax_2019'),
                 ('OD_PAX','y_log'),('OD_REV_USD','y_rev_log'),('rev_lag_24','log_rev_lag_24')]:
    df_p[d_col] = np.log1p(np.clip(df_p[s], 0, None)).astype('float32')
df_p['y_fare_log'] = np.log1p(np.clip(df_p['avg_fare'], 1.0, None)).astype('float32')
df_p['log_fare_lag_24'] = np.log1p(np.clip(df_p['fare_lag_24'], 1.0, None)).astype('float32')

df_p['lag24_pct_rank'] = df_p.groupby('DEP_YEAR')['log_pax_lag_24'].rank(pct=True).astype('float32')
df_p['lag24_zscore'] = df_p.groupby('DEP_YEAR')['log_pax_lag_24'].transform(lambda s: (s - s.mean()) / (s.std() + 1e-6)).astype('float32')
df_p['season_ratio_lag24'] = (df_p['log_pax_lag_24'] - df_p['log_pax_yravg_lag2']).astype('float32')
df_p['mom_1m_lag24'] = (df_p['log_pax_lag_24'] - df_p['log_pax_lag_25']).astype('float32')
df_p['ratio_vs_2019_lag24'] = (df_p['log_pax_lag_24'] - df_p['log_pax_2019']).fillna(0).astype('float32')
raw_yoy_p = df_p['log_pax_lag_24'] - df_p['log_pax_lag_36']
df_p['yoy_growth_z'] = raw_yoy_p.groupby(df_p['DEP_YEAR']).transform(lambda s: (s - s.mean()) / (s.std() + 1e-6)).astype('float32')
df_p['fare_lag24_z'] = df_p.groupby('DEP_YEAR')['log_fare_lag_24'].transform(lambda s: (s - s.mean()) / (s.std() + 1e-6)).astype('float32')

ps_p = ap_pop_lp[['AIRPRT_CD','DEP_YEAR','pop','pop_g']]
df_p = df_p.merge(ps_p.rename(columns={'AIRPRT_CD':'ORG_AIRPRT','pop':'org_pop','pop_g':'org_pop_growth'}), on=['ORG_AIRPRT','DEP_YEAR'], how='left')
df_p = df_p.merge(ps_p.rename(columns={'AIRPRT_CD':'DES_AIRPRT','pop':'des_pop','pop_g':'des_pop_growth'}), on=['DES_AIRPRT','DEP_YEAR'], how='left')
df_p['log_gravity'] = np.log1p((df_p['org_pop'].fillna(0)*df_p['des_pop'].fillna(0))/(np.clip(df_p['distance_mi'],50,None)**2)).astype('float32')

fs_p = faa_p[['AIRPRT_CD','DEP_YEAR','hub','enp']]
df_p = df_p.merge(fs_p.rename(columns={'AIRPRT_CD':'ORG_AIRPRT','hub':'org_faa_hub','enp':'org_faa_enp'}), on=['ORG_AIRPRT','DEP_YEAR'], how='left')
df_p = df_p.merge(fs_p.rename(columns={'AIRPRT_CD':'DES_AIRPRT','hub':'des_faa_hub','enp':'des_faa_enp'}), on=['DES_AIRPRT','DEP_YEAR'], how='left')
df_p['log_org_faa_enp'] = np.log1p(df_p['org_faa_enp']).astype('float32')
df_p['log_des_faa_enp'] = np.log1p(df_p['des_faa_enp']).astype('float32')

tr_p = (df_p['t'] <= max_t) & (df_p['pax_lag_24'].notna())
fu_p = (df_p['t'] > max_t) & (df_p['pax_lag_24'].notna())

y_tr_deb_p = weighted_debias_target(df_p.loc[tr_p])
sw_tr_p = np.sqrt(np.clip(df_p.loc[tr_p, 'pax_lag_24'].values, 1.0, None))
hgb_pax_prod = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_depth=7, min_samples_leaf=80, l2_regularization=5.0, random_state=42)
hgb_pax_prod.fit(df_p.loc[tr_p, INV_EXT], y_tr_deb_p, sample_weight=sw_tr_p)
smear_prod = float(np.average(np.exp(y_tr_deb_p - hgb_pax_prod.predict(df_p.loc[tr_p, INV_EXT])), weights=sw_tr_p))

p_hgb_fu = np.expm1(np.clip(df_p.loc[fu_p, 'log_pax_lag_24'].values + hgb_pax_prod.predict(df_p.loc[fu_p, INV_EXT]), 0, 15)) * smear_prod
p_ens_fu = 0.70 * p_hgb_fu + 0.30 * df_p.loc[fu_p, 'pax_lag_24'].values

f_tr_p = tr_p & (df_p['fare_lag_24'].notna()) & (df_p['avg_fare']>20) & (df_p['avg_fare']<5000)
yf_deb_p = weighted_debias_target(df_p.loc[f_tr_p], y_col='y_fare_log', lag_col='log_fare_lag_24', w_col='pax_lag_24', clip_val=0.5)
hgb_fare_prod = HistGradientBoostingRegressor(max_iter=150, learning_rate=0.05, max_depth=6, min_samples_leaf=80, l2_regularization=5.0, random_state=42)
hgb_fare_prod.fit(df_p.loc[f_tr_p, INV_EXT], yf_deb_p, sample_weight=np.sqrt(np.clip(df_p.loc[f_tr_p, 'pax_lag_24'].values, 1.0, None)))
p_fare_fu = 0.75 * np.expm1(np.clip(df_p.loc[fu_p, 'log_fare_lag_24'].fillna(5.5).values + hgb_fare_prod.predict(df_p.loc[fu_p, INV_EXT]), 2, 9)) + 0.25 * df_p.loc[fu_p, 'fare_lag_24'].fillna(250.0).values

yr_deb_p = weighted_debias_target(df_p.loc[f_tr_p], y_col='y_rev_log', lag_col='log_rev_lag_24', w_col='rev_lag_24', clip_val=1.0)
hgb_rev_prod = HistGradientBoostingRegressor(max_iter=150, learning_rate=0.05, max_depth=6, min_samples_leaf=80, l2_regularization=5.0, random_state=42)
hgb_rev_prod.fit(df_p.loc[f_tr_p, INV_EXT], yr_deb_p, sample_weight=np.sqrt(np.clip(df_p.loc[f_tr_p, 'rev_lag_24'].values, 1.0, None)))
p_rev_direct_fu = np.expm1(np.clip(df_p.loc[fu_p, 'log_rev_lag_24'].fillna(10.0).values + hgb_rev_prod.predict(df_p.loc[fu_p, INV_EXT]), 0, 20))
naive_rev_fu = df_p.loc[fu_p, 'pax_lag_24'].values * df_p.loc[fu_p, 'fare_lag_24'].fillna(250.0).values
p_rev_fu = 0.50 * p_rev_direct_fu + 0.35 * (p_ens_fu * p_fare_fu) + 0.15 * naive_rev_fu

out_fc = df_p.loc[fu_p, ['ORG_AIRPRT','DES_AIRPRT','DEP_YEAR','DEP_MONTH','distance_mi','is_domestic','is_long_haul_intl','pax_lag_24','fare_lag_24']].copy()
out_fc['pred_od_pax'] = np.round(p_ens_fu, 1)
out_fc['pred_avg_fare_usd'] = np.round(p_fare_fu, 2)
out_fc['pred_od_rev_usd'] = np.round(p_rev_fu, 0)
out_fc.to_csv('AA_2026_2028_24mo_forecast_v3.csv', index=False)

y27 = out_fc[out_fc['DEP_YEAR']==2027].groupby(['ORG_AIRPRT','DES_AIRPRT','is_domestic','is_long_haul_intl'], as_index=False).agg(
    base_pax_2025=('pax_lag_24','sum'), pred_pax_2027=('pred_od_pax','sum'),
    pred_rev_2027_M=('pred_od_rev_usd', lambda s: s.sum()/1e6), avg_fare_2027=('pred_avg_fare_usd','mean')
)
y27['pax_growth_pct'] = (y27['pred_pax_2027'] - y27['base_pax_2025']) / np.maximum(y27['base_pax_2025'], 1) * 100
top15 = y27.sort_values('pred_rev_2027_M', ascending=False).head(15)
top15.to_csv('AA_top_15_markets_2027_v3.csv', index=False)

print(f'Production forecast saved in {time.time()-t_prod:.1f}s | {len(out_fc):,} route-months forecasted (Aug 2026 - Jul 2028)')
print(f"2027 System Forecast: {y27['pred_pax_2027'].sum()/1e6:.2f}M PAX | ${y27['pred_rev_2027_M'].sum()/1e3:.2f}B O&D Revenue")
print(top15.head(10).round(2).to_string(index=False))
